# Introduction to optimisation

Numerical optimisation is an area of mathematics and computer science concerned with finding the best (optimal) solution to a given problem. Most often, this means finding the minimum or maximum of an objective function (e.g. minimising costs, maximising profit, or minimising the error of a machine-learning model).

Lecture outline

1. Searching for an optimum using naive (brute-force) search with a single loop.
2. Why does adding a second dimension (a nested loop) significantly increase computational complexity?
3. Optimisation in practice: using dedicated algorithms to search for a solution.
4. Fuzzy optimisation, where the objective is not crisp.
5. Representing chemical processes in Python.
6. Optimising chemical processes.

In [ ]:
from chempy import balance_stoichiometry
from chempy import Substance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import pyomo.environ as pyo
from pyomo.common.numeric_types import RegisterIntegerType
from sympy.core.numbers import Integer, Number, Zero, One

In [ ]:
from scipy.optimize import minimize, minimize_scalar

In [ ]:
from tqdm import tqdm

In [ ]:
RegisterIntegerType(Integer)
RegisterIntegerType(Number)
RegisterIntegerType(Zero)
RegisterIntegerType(One)

## Naive (brute-force) search: a single loop

Function to be minimised

$f(x)=(x−3)^2 + 5$

What is the minimum of this function according to the formula?

In [ ]:
def func_1d(x: float) -> float:
    return (x - 3)**2 + 5

How can a computer solve this problem? Searching for the minimum can be reduced to an exhaustive check of all possible values of the parameter $x$. We must, however, make some initial assumptions about the search range.

In [ ]:
# Search space: from -100 to 100, with a step of 0.001
x_values = np.arange(-100, 100, 0.001)

Then, one value at a time, we can examine how the value returned by the optimised function changes.

In [ ]:
%%time

y_values = []
best_x = None
min_value = float("inf")

# Search all values one by one
for x in x_values:
    y = func_1d(x)
    y_values.append(y)

    if y < min_value:
        min_value = y
        best_x = x

print(f"Minimum found: {min_value:.2f} at x = {best_x:.2f}")
print(f"Number of points checked: {len(x_values)}")

Visualisation can be a useful supporting tool for examining the behaviour of a function.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(x_values, y_values)
plt.plot(best_x, min_value, "x")

plt.title("One-parameter function")
plt.xlabel("Function parameter value")
plt.ylabel("Function value")
plt.grid(True)
plt.show()

## Naive search: a nested loop

Now consider a function of two parameters, for example a simplified cost model depending on the raw-material price $x$ and the production time $y$:

$f(x,y)=(x−2)^2 + (y+1)^2$

Under the same assumptions as above, how many variants do we need to examine?

In [ ]:
len(np.arange(-100, 100, 0.001)) ** 2

It is worth increasing the step size, which reduces the number of points in each dimension by a factor of 10.

In [ ]:
len(np.arange(-100, 100, 0.01)) ** 2

In [ ]:
def fun_2d(x: float, y: float) -> float:
    return (x - 2)**2 + (y + 1)**2

In [ ]:
x_values_2d = np.arange(-100, 100, 0.01)
y_values_2d = np.arange(-100, 100, 0.01)

We are therefore dealing with a function of two independent parameters: $x$ and $y$. A single loop is no longer sufficient to find the optimum. Because the two parameters are independent, we need a separate loop for each of them so that all possible combinations of their values can be tested.

For a function of one parameter, the number of variants to check was simply the number of values included in the search range. With two parameters, two nested loops form the Cartesian product of the values from both loops. Hence, the total number of operations is the square of the size of a single loop.

In such situations we speak of linear and quadratic computational complexity, corresponding to the number of operations required to find the optimum. This is related to the so-called curse of dimensionality: as the number of parameters increases, complexity grows exponentially with the dimension.

> Computational complexity is formally expressed using asymptotic notation, which describes how an algorithm's resource requirements grow with the size of the problem ($N$). The notation $\mathcal{O}$ gives an upper bound (pessimistic cost), $\Omega$ a lower bound (optimistic cost), and $\Theta$ a tight bound when the algorithm behaves in the same asymptotic way in every case. In naive (brute-force) optimisation with two parameters, every one of the $N\times N$ points must be checked, with no possibility of terminating the loops early. Therefore the same number of operations is performed in both the pessimistic and optimistic cases, giving a tight time complexity of $\Theta(N^2)$.
>
> When discussing resources, however, time must be distinguished from memory. The time complexity is $\Theta(N^2)$ because the theoretical running time grows proportionally to the square of the number of samples. Memory complexity depends on the implementation: if the grid is traversed using two nested loops and only the best value found so far (minimum/maximum) is stored, the algorithm uses a constant amount of memory, giving $\Theta(1)$ space complexity. Memory usage would grow to $\Theta(N^2)$ only if the algorithm first generated and stored every grid point and its result before selecting the best one.

In [ ]:
%%time

z_values = []
best_xy = None
min_value_2d = float("inf")
iteration_count = 0

for x in tqdm(x_values_2d):
    for y in y_values_2d:
        z = fun_2d(x, y)
        z_values.append(z)
        iteration_count += 1

        if z < min_value_2d:
            min_value_2d = z
            best_xy = (x, y)

print(f"Minimum found: {min_value_2d:.4f} at x = {best_xy[0]:.2f}, y = {best_xy[1]:.2f}")
print(f"Number of points checked: {iteration_count}")

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

X, Y = np.meshgrid(x_values_2d[::10], y_values_2d[::10])
Z = fun_2d(X, Y)

surf = ax.plot_surface(
    X,
    Y,
    Z,
    cmap="coolwarm",
    linewidth=0,
    antialiased=False,
)

ax.zaxis.set_major_formatter("{x:.02f}")

fig.colorbar(surf, shrink=0.5, aspect=5)
plt.xlabel("X")
plt.ylabel("Y")
ax.set_zlabel("Function value")

plt.show()

## Optimisation in practice

Optimising a two-parameter function using the naive method took **more than 2 minutes** (on an Apple M1 Pro). This is a consequence of the curse of dimensionality discussed above. What happens when we have a function of three parameters, or even more? Modern neural networks of moderate size commonly have millions of parameters.

In practice, we therefore use algorithmic numerical optimisation methods whose primary goal is to reduce the computational complexity of the search for optimal solutions. The designers of such algorithms have repeatedly drawn inspiration from processes observed in everyday physical and biological phenomena. In many cases, the algorithms are also based purely on mathematics. We distinguish white-box, grey-box, and black-box approaches, whose operation depends completely, partially, or not at all on the formula (or, more generally, the internal structure) of the function being optimised.

How quickly can a dedicated algorithm (BFGS from the SciPy library) find the optimum of a two-parameter function?

> BFGS (named after its authors: Broyden, Fletcher, Goldfarb, and Shanno) is one of the most popular unconstrained optimisation algorithms and belongs to the class of quasi-Newton methods. Unlike the classical Newton method, which requires the expensive computation and inversion of the full matrix of second derivatives (the Hessian) at every step, BFGS iteratively approximates the inverse Hessian using the history of changes in the gradient $\nabla f(x)$. As a result, it combines convergence that is almost as fast (superlinear) as that of second-order methods with a relatively low computational cost close to that of first-order methods. This makes it a common default choice in numerical optimisation libraries for smooth functions with a small or moderate number of parameters.

In [ ]:
%%time

# Start the search from an arbitrary point (e.g. x=50, y=50)
start_coords = (50, 50)

# Optimisation algorithm from SciPy
result = minimize(lambda v: fun_2d(v[0], v[1]), start_coords, method="BFGS")

In [ ]:
print(f"Minimum found: {result.fun:.2f} at x = {result.x[0]:.2f}, y = {result.x[1]:.2f}")
print(f"Number of function evaluations: {result.nfev}")

Instead of millions of iterations, the algorithm required only a few dozen function evaluations!

## Fuzzy optimisation

In classical optimisation, the objective is crisp (for example, minimise $f(x, y, ...)$. In real-world problems, objectives are often "fuzzy". When buying a car, for example, we might have two objectives that are not expressed as strict thresholds: the price should be relatively low and the mileage should not be too high.

In fuzzy optimisation (based on fuzzy logic), rather than the 0 and 1 values known from classical logic (false/true), we use a degree of membership ranging from 0.0 to 1.0. Fuzzy optimisation consists of maximising the degree of satisfaction of all, often conflicting, objectives simultaneously.

Below is a simple example. The aim is to find an operating point for a machine (decision variable: temperature $T$) such that performance is relatively high while energy cost remains acceptably low.

In [ ]:
# Decision variable: temperature in degrees Celsius
T = np.linspace(20, 100, 500)

In [ ]:
# 1. Membership function for "Performance is high" 
# (Performance increases with temperature and reaches its optimum between 40°C and 80°C)
performance_satisfaction = np.clip((T - 40) / 40, 0, 1)

In [ ]:
# 2. Membership function for "Cost is low"
# (Energy costs increase sharply above 60°C and are unacceptable at 90°C)
cost_satisfaction = np.clip((90 - T) / 30, 0, 1)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(T, performance_satisfaction, label="Objective 1: High performance", color="blue", linestyle="--")
plt.plot(T, cost_satisfaction, label="Objective 2: Low cost", color="red", linestyle="--")

plt.title("Performance vs. cost")
plt.xlabel("Temperature (°C)")
plt.ylabel("Degree of satisfaction (0–1)")
plt.legend()
plt.grid(True)
plt.show()

The solution is to use the Bellman–Zadeh method, which aggregates the membership functions for both criteria using the minimum operator. The optimal decision is the point at which the two values achieve the best compromise, i.e. where the two curves intersect.

$$ \mu_D (T) = \mu_P (T) \land \mu_C (T) = \min(\mu_P (T), \mu_C (T)) $$

- $T$ – decision variable: temperature
- $P$ – parameter: satisfaction with performance
- $C$ – parameter: satisfaction with cost
- $\mu$ – membership function for the parameters at a given value of the decision variable

In [ ]:
total_satisfaction = np.minimum(performance_satisfaction, cost_satisfaction)

Finding the optimal temperature consists of identifying the point with the maximum value of the aggregated membership function obtained by taking the minimum of the two individual membership functions.

In [ ]:
opt_idx = np.argmax(total_satisfaction)
opt_T = T[opt_idx]
max_zadowolenie = total_satisfaction[opt_idx]

In [ ]:
# Visualisation
plt.figure(figsize=(10, 5))
plt.plot(T, performance_satisfaction, label="Objective 1: High performance", color="blue", linestyle="--")
plt.plot(T, cost_satisfaction, label="Objective 2: Low cost", color="red", linestyle="--")
plt.plot(T, total_satisfaction, label="Overall satisfaction (min-max)", color="green", linewidth=2)

plt.axvline(opt_T, color="purple", linestyle=":", label=f"Fuzzy optimum: {opt_T:.1f}°C")
plt.title("Fuzzy optimisation: compromise between performance and cost")
plt.xlabel("Temperature (°C)")
plt.ylabel("Degree of satisfaction (0–1)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(f"The best compromise (fuzzy optimum) was obtained for Temperature = {opt_T:.1f}°C")
print(f"The degree of satisfaction with the solution is: {max_zadowolenie:.2f} (on a 0–1 scale)")

## Representing chemical processes in Python

One popular solution for representing chemical processes in Python is the [ChemPy](https://github.com/bjodah/chempy) library, which can be installed with `pip install chempy`.

The process used in the next few examples is methane combustion.

Reactants:
- methane (CH4)
- oxygen (O2)

Products:
- carbon dioxide (CO2)
- water (H2O)

### How to determine the proportions of reactants and products

The **ChemPy** library provides the `balance_stoichiometry` function, which automatically balances the proportions for a specified reaction.

In [ ]:
substrates = {"CH4", "O2"}
products = {"CO2", "H2O"}

In [ ]:
substrates_proportions, products_proportions = balance_stoichiometry(substrates, products)

The following stoichiometric proportions are obtained for the reactants:

In [ ]:
for item, amount in substrates_proportions.items():
    print(f"- {item} x {amount}")

And for the products:

In [ ]:
for item, amount in products_proportions.items():
    print(f"- {item} x {amount}")

This can ultimately be written as: $CH_4 + 2O_2 \to CO_2 + 2H_2O$

### How much does it all weigh?

The **ChemPy** library also provides functionality for determining the absolute masses of substances from their chemical formulae.

In [ ]:
methane = Substance.from_formula("CH4")
water = Substance.from_formula("H2O")

In [ ]:
print(f"1 mole of methane (CH4) weighs: {methane.mass:.4f} grams.")
print(f"1 mole of water (H2O) weighs: {water.mass:.4f} grams.")

## Optimising a chemical process

We have a cylinder with a capacity of 100 moles of gas. The objective is to find a mixture of methane and oxygen that maximises the relative combustion product.

The function that determines the combustion product can be defined as follows:

In [ ]:
required_methane = substrates_proportions["CH4"]
required_oxygene = substrates_proportions["O2"]

In [ ]:
product_sum = sum(products_proportions.values())

In [ ]:
def f_opt(methane_amount: int, oxygene_amount: int) -> int:
    methane_cycles = methane_amount // required_methane
    oxygene_cyxles = oxygene_amount // required_oxygene
    real_cycles = np.minimum(methane_cycles, oxygene_cyxles)
    total_product = real_cycles * product_sum

    return total_product

### Naive optimisation

Using naive optimisation means checking every possible gas-mixture combination:
- 1× methane, 99× oxygen
- 2× methane, 98× oxygen
- ...
- 99× methane, 1× oxygen

The final result is the combination that produces the largest amount of combustion product.

In [ ]:
%%time

total_capacity = 100

max_product = 0
optimal_methane = 0
optimal_oxygene = 0

# Loop through every option (from 1 to 99 portions of methane)
for methane_amount in range(1, total_capacity):
    # Fill the remaining cylinder capacity with oxygen
    oxygene_amount = total_capacity - methane_amount

    current_total_product = f_opt(methane_amount, oxygene_amount)

    # If this variant is better than the best one found so far, store it
    if current_total_product > max_product:
        max_product = current_total_product
        optimal_methane = methane_amount
        optimal_oxygene = oxygene_amount

In [ ]:
print(f"Methane: {optimal_methane} moles")
print(f"Oxygen: {optimal_oxygene} moles")
print(f"The mixture can produce a total of {max_product:.1f} moles")

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})


X, Y = np.meshgrid(np.linspace(0, 100, 50), np.linspace(0, 100, 50))
X_mod = np.where(X + Y > 100, 100 - Y, X)
Y_mod = np.where(X + Y > 100, 100 - X, Y)
Z = f_opt(X_mod, Y_mod)

surf = ax.plot_surface(
    X,
    Y,
    Z,
    cmap="coolwarm",
    linewidth=0,
    antialiased=False,
)

# ax.zaxis.set_major_locator(LinearLocator(10))
ax.zaxis.set_major_formatter("{x:.02f}")

fig.colorbar(surf, shrink=0.5, aspect=5)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Produkty spalania")

plt.show()

### Algorithmic optimisation

#### The SciPy library

The [SciPy](https://scipy.org/) library provides many implementations of numerical optimisation algorithms. There is, however, one limitation in the interface used here: the objective function is minimised. This requires a small modification of the original objective function when our actual goal is maximisation.

In [ ]:
def f_maximize(methane_amount: int) -> int:
    oxygene_amount = 100 - methane_amount
    methane_cycles = methane_amount // required_methane
    oxygene_cyxles = oxygene_amount // required_oxygene
    real_cycles = np.minimum(methane_cycles, oxygene_cyxles)
    total_product = real_cycles * product_sum

    return total_product

In [ ]:
def f_target(x: float) -> float:
    arg = round(float(np.asarray(x).item()))

    result_int = f_maximize(arg)

    # For maximisation, negate the result
    return -float(np.asarray(result_int).item())

In [ ]:
result = minimize_scalar(f_target, bounds=(0, 100), method="bounded")

In [ ]:
print(f"Methane: {result.x.item():.01f} moles")
print(f"Oxygen: {100 - result.x.item():.01f} moles")
print(f"The mixture can produce a total of {-result.fun.item():.1f} moles")

#### The Pyomo library

In practice, [Pyomo](https://www.pyomo.org/) is often a more natural choice because it is a dedicated framework for optimisation. Unlike **SciPy**, in **Pyomo** the objective function is defined directly using Pyomo objects. This approach enables a clear and unambiguous definition of the operations that will later be analysed directly by automated optimisation tools.

First, create a new object representing the mathematical model of the gas cylinder.

In [ ]:
model = pyo.ConcreteModel()

Then define the model variables and parameters.

In [ ]:
model.M = pyo.Var(within=pyo.NonNegativeIntegers, bounds=(0, total_capacity))  # Amount of methane
model.O = pyo.Var(within=pyo.NonNegativeIntegers, bounds=(0, total_capacity))  # Amount of oxygen
model.C = pyo.Var(within=pyo.NonNegativeIntegers)  # Number of complete reaction cycles

Next, define the objective.

In [ ]:
model.target = pyo.Objective(expr=model.C * float(product_sum), sense=pyo.maximize)

The cylinder has a capacity of 100 moles. Overfilling it could cause an explosion, so constraints must be introduced to preserve the appropriate proportions of the gas mixture.

In [ ]:
model.capacity = pyo.Constraint(expr=model.M + model.O == total_capacity)

The number of reaction cycles also cannot exceed the available amount of gas.

In [ ]:
# This replaces: cycles = gas_amount // required_gas
model.methane_constraint = pyo.Constraint(expr=model.C * required_methane <= model.M)

In [ ]:
model.oxygene_constraint = pyo.Constraint(expr=model.C * required_oxygene <= model.O)

Optimisation

To perform the actual optimisation, we need a **solver**: a program or mathematical procedure used to find solutions to equations, systems of equations, and optimisation problems. The term *solver* is often associated with the optimisation add-in in MS Excel, but in practice solvers are more commonly encountered as processes running "under the hood" to solve optimisation problems.

To search for the optimal gas mixture, we will use the free [GLPK](https://en.wikipedia.org/wiki/GNU_Linear_Programming_Kit) solver. It must be installed separately using a command appropriate for the environment:
- macOS: `brew install glpk`
- Bash / Google Colab: `!apt-get install -y -qq glpk-utils`
- Conda: `conda install -c conda-forge glpk`

In [ ]:
solver = pyo.SolverFactory("glpk")
result = solver.solve(model)

The results — both the optimal gas-mixture parameters and the maximum combustion product — are available directly from the mathematical model that we created.

In [ ]:
methane_result = pyo.value(model.M)
oxygene_result = pyo.value(model.O)
cycles = pyo.value(model.C)
product_result = pyo.value(model.target)

In [ ]:
print(f"Methane: {methane_result:.01f} moles")
print(f"Oxygen: {oxygene_result:.01f} moles")
print(f"The mixture can produce a total of {product_result:.1f} moles")

### Fuzzy optimisation

To find the optimal gas mixture we will again use the Bellman–Zadeh method. First, define the parameters while also adding a constraint representing the maximum cylinder capacity.

In [ ]:
range_ = np.arange(0, total_capacity + 1)
M, O = np.meshgrid(range_, range_)

In [ ]:
V = M + O
mu_constraint = np.where(V <= total_capacity, 1.0, 0.0)

Next, determine the combustion product for every combination of gas proportions.

In [ ]:
products = f_opt(M, O)

We also need the maximum product value in order to scale the actual product to a membership value in the range 0.0–1.0.

In [ ]:
max_products = (total_capacity // (required_methane + required_oxygene)) * product_sum

In [ ]:
mu_target = products / max_products

As in the previous example, finding the optimal value reduces to aggregation using the $min$ function.

In [ ]:
mu_optimum = np.minimum(mu_target, mu_constraint)

In [ ]:
opt_idx = np.unravel_index(np.argmax(mu_optimum), mu_optimum.shape)
opt_methane = M[opt_idx]
opt_oxygene = O[opt_idx]
opt_products = products[opt_idx]

In [ ]:
print(f"Methane: {opt_methane.item()} moles")
print(f"Oxygen: {opt_oxygene.item()} moles")
print(f"The mixture can produce a total of {opt_products} moles")

#### Visualisation

In [ ]:
plt.figure(figsize=(10, 8))
plt.contourf(M, O, mu_optimum.astype(float), levels=50, cmap="viridis")
plt.colorbar(label="Fuzzy decision-function value (\u03bc)")

x_lin = np.linspace(0, 100, 100)
y_lin = 100 - x_lin
plt.plot(x_lin, y_lin, color="red", linestyle="--", linewidth=2, label="Capacity constraint ($CH_4+O_2=100$)")

plt.scatter(opt_methane, opt_oxygene, color="red", s=100, edgecolor="black", zorder=5, 
            label=f"Optimum ({opt_methane} $CH_4$, {opt_oxygene} $O_2$)")

plt.title("Fuzzy optimisation of methane combustion")
plt.xlabel("Amount of methane in the cylinder")
plt.ylabel("Amount of oxygen in the cylinder")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Popular real-world applications of optimisation

### Supply chain

Imagine a retail chain with 3 warehouses and 50 shops. The objective is to deliver goods to the shops while minimising transport costs and, at the same time, preventing stock-outs.

1. Demand prediction using machine learning. The model analyses historical data, weather, day of the week, and upcoming holidays to generate a forecast such as: "Shop 1 in Gdańsk will sell 120 pallets of water tomorrow, while Shop 2 in Kraków will sell 85 pallets."
2. Define the decision variables. The optimisation system must know what it is allowed to control. Here, the variables are the quantities of goods shipped from each warehouse to each shop.
3. Define the objective function.
   1. This is the "compass" for the entire process. The objective function is a mathematical equation describing the total transport cost.
   2. Total cost = (quantity from Warehouse 1 to Shop 1 $\times$ price per km) + (quantity from Warehouse 1 to Shop 2 $\times$ price per km) + ...
4. Add constraints, which are common in such processes.
   1. Supply constraints: Warehouse 1 contains only 500 pallets. The total quantity shipped from this warehouse cannot exceed 500.
   2. Demand constraints: Shop 1 must receive exactly 120 pallets (the quantity predicted by the predictive model).
   3. No negative shipments: we cannot ship, for example, -10 pallets, so all variables must satisfy $\ge 0$.
5. Run the optimisation algorithm (solver).

### Dynamic pricing

This is a classic problem in airlines, taxis, hotels, and e-commerce. Prices are continually adjusted in order to maximise profit.

1. Predict demand and price elasticity using machine learning.
   1. This is where machine learning enters the process (for example, a regression model).
   2. The model analyses historical data, booking lead time, day of the week, seasonality, and competitors' prices to estimate a demand function such as: "If we set the price at PLN 1,500, the probability of purchase will be 85%, but at PLN 2,200 it will fall to 30%."
2. Define the decision variables.
   1. The optimisation system must know what it is allowed to control.
   2. Here, the variables are ticket-price levels in different time periods or for specific seat allocations.
3. Define the objective function.
   1. The objective function is a mathematical equation describing the total revenue from selling all seats.
   2. Total revenue = (price in period 1 × expected number of tickets sold in period 1) + (price in period 2 × expected number of tickets sold in period 2) + ...
4. Add constraints.
   1. Capacity (supply) constraints: the aircraft has a fixed number of seats (e.g. 180). The total number of tickets sold across all periods cannot exceed 180.
   2. Business and market constraints: the price cannot fall below a minimum threshold (e.g. PLN 800 to cover costs) or rise above a legal/reputational limit (e.g. PLN 5,000).
   3. No negative prices or sales: prices and the number of seats sold must be non-negative.
5. Run the optimisation algorithm (solver). The solver — often a nonlinear-programming solver because of the demand curve — searches the decision space and determines the optimal price trajectory over time, yielding the highest possible profit.

### Task or route allocation optimisation (VRP — Vehicle Routing Problem)

Imagine a courier company with 100 parcels to deliver and 5 couriers.

1. Predict travel times and delivery windows using machine learning.
   1. This is where machine learning enters the process (for example, a regression model).
   2. The model analyses historical GPS data, traffic volume, time of day, weather, and location-specific characteristics (e.g. parking availability) to generate a forecast such as: "Travelling from point A to point B at 14:30 will take 18 minutes, and serving the customer at this address will take 4 minutes."
2. Define the decision variables.
   1. The optimisation system must know what it is allowed to control (we may have seen this before...).
   2. Here, the variables are binary decisions selecting connections between locations for particular vehicles, together with the order and arrival times for individual customers.
3. Define the objective function.
   1. The objective function is a mathematical equation describing the total cost of all routes.
   2. Total cost = (total distance travelled × fuel cost per km) + (total courier working time × hourly rate) + penalties for any deliveries outside the specified time windows.
4. Add constraints.
   1. Capacity constraints: each van has a weight/volume limit (e.g. at most 30 parcels or 400 kg). The total load on a courier's route cannot exceed this value.
   2. Service and route-continuity constraints: each of the 100 customers must be visited exactly once by one courier, and every driver's route must start and finish at the depot (main warehouse).
   3. Working-time constraints (time windows): a single route cannot exceed an 8-hour working day, and consignments must arrive within the time windows declared to customers.
5. Run the optimisation algorithm. Because the number of possible combinations is enormous (the problem is NP-hard), a solver can use advanced metaheuristics (e.g. genetic algorithms, simulated annealing, or Large Neighbourhood Search) to find a near-optimal route in minutes, avoiding the need to inspect billions of permutations.